# pandas — Practices: Selection, Merge, GroupBy, Bucketing, Transform

This notebook demonstrates key pandas techniques from THEORY.md with small, finance-focused examples, followed by exercises at the end.

In [ ]:
import pandas as pd
import numpy as np
pd.options.display.width = 120

# sample data: invoices / transactions
df = pd.DataFrame({
    'invoice': [101,102,103,104,105,106],
    'account': ['A','A','B','B','C','C'],
    'date': ['2024-01-05','2024-01-07','2024-02-01','2024-02-15','2024-03-01',None],
    'amount': [120.0, -5.0, 500.00, 500.0, 1200.5, 0],
    'currency': ['USD','USD','USD','EUR','USD','USD'],
    'customer': ['Acme Inc','Beta LLC','Acme Inc','Delta Co','Echo Ltd','Gamma SA']
})
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df

## Selection examples (boolean masks, .loc, .isin, between, str.contains)

In [ ]:
# boolean mask
df_positive = df[df['amount'] > 0]
display(df_positive)

# .loc assignment (avoid chained assignment)
df.loc[df['amount'] <= 0, 'amount'] = 0
display(df)

# .isin and .between
display(df[df['account'].isin(['A','B'])])
display(df[df['date'].between('2024-01-01','2024-02-28')])

# string contains
display(df[df['customer'].str.contains('Inc', na=False)])

## Indexing & time-based selection (DatetimeIndex, resample)

In [ ]:
# set datetime index and resample by month
df_time = df.set_index('date')
# resample monthly sums (NaT rows dropped automatically)
monthly = df_time['amount'].resample('M').sum()
monthly

## Merge / reconciliation example (indicator & validate)

In [ ]:
# create another table (payments) to merge on invoice
payments = pd.DataFrame({
    'invoice':[101,103,105,107],
    'paid': [120.0, 500.0, 1200.5, 50.0]
})
merged = pd.merge(df, payments, on='invoice', how='outer', indicator=True, suffixes=('_inv','_pay'))
display(merged)
# check unmatched rows
display(merged['_merge'].value_counts())

## GroupBy, agg and transform examples

In [ ]:
# groupby agg
grp = df.groupby('account').agg(total_amount=('amount','sum'), txn_count=('amount','count'))
display(grp)

# transform: percent of group
df['group_sum'] = df.groupby('account')['amount'].transform('sum')
df['pct_of_account'] = df['amount'] / df['group_sum']
display(df)

## Bucketing: pd.cut and pd.qcut

In [ ]:
bins = [0, 100, 500, 1000, float('inf')]
labels = ['small','medium','large','xlarge']
df['amount_bucket'] = pd.cut(df['amount'], bins=bins, labels=labels, right=False, include_lowest=True)
display(df[['invoice','amount','amount_bucket']])
# quantile buckets
df['amount_qbucket'] = pd.qcut(df['amount'].fillna(0), q=4, labels=['Q1','Q2','Q3','Q4'])
display(df[['invoice','amount','amount_qbucket']])

## Transform vs Apply demo (performance and replacement patterns)

In [ ]:
import time
# build larger sample to time apply vs transform
big = pd.concat([df]*1000, ignore_index=True)
start = time.time()
s = big.groupby('account')['amount'].transform(lambda s: s / s.sum())
t_apply = time.time() - start
start = time.time()
s2 = big['amount'] / big.groupby('account')['amount'].transform('sum')
t_transform = time.time() - start
t_apply, t_transform

## Exercises
1. Load a CSV of transactions (create a small CSV if needed) and replicate the bucket grouping.
2. Merge transactions with payments and produce a reconciliation report showing unmatched invoices.
3. Compute percent of account totals and identify transactions contributing to top 80% of each account (Pareto).
4. Replace the `apply` normalization above with a `transform` approach and measure timings.
5. Handle NaN dates: set them to the last day of the previous month and resample monthly totals.